# gama-rca: dev-30 arms
Ubah hanya `ARM` di Cell 4. Cell 3, 5, 6 dibaca dari repo (commit yang di-clone di Cell 1). GPU, Internet ON, Secret GITHUB_TOKEN, input dataset dan model.

In [ ]:
# %% Cell 1 -- code (private repo) and data
import os, subprocess

REPO = "solaemanachmad/gama-rca"                      # private; no leading slash
DATA = "/kaggle/input/datasets/solaemanachmad/rca100v1-1/RCA100v1.1"   # folder that contains cases/ and manifest.txt
WORK = "/kaggle/working/gama-rca"

from kaggle_secrets import UserSecretsClient          # Add-ons > Secrets: GITHUB_TOKEN (read-only token for the private repo)
tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
subprocess.run(["rm", "-rf", WORK])
r = subprocess.run(["git", "clone", "--depth", "60", f"https://{tok}@github.com/{REPO}.git", WORK],
                   capture_output=True, text=True)
print("clone rc =", r.returncode, (r.stderr or "").replace(tok, "***")[-300:])
os.chdir(WORK)
COMMIT = subprocess.run("git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
print("COMMIT", COMMIT); print(subprocess.run("git log --oneline | head -5", shell=True, capture_output=True, text=True).stdout)
assert os.path.isdir(os.path.join(DATA, "cases")), f"dataset path wrong: {DATA}"
os.environ["RCA100_ROOT"] = DATA
os.environ["PYTHONHASHSEED"] = "0"

In [ ]:
# %% Cell 2 -- dependencies (skip what the notebook image already has)
import sys, importlib
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "sentence-transformers", "networkx", "drain3", "faiss-cpu", "langgraph"])
for m in ("drain3", "faiss", "sentence_transformers", "networkx", "langgraph", "torch", "transformers"):
    try:
        importlib.import_module(m); print("ok  ", m)
    except Exception as e:
        print("MISSING", m, "->", e)

In [ ]:
# Cell 3: dibaca dari repo hasil clone, jadi selalu sesuai commit (arm baru tidak perlu notebook baru)
src = open(os.path.join(WORK, "scripts", "kaggle_run.py"), encoding="utf-8").read()
exec(compile(src.split("# %% Cell 3")[1].split("\n", 1)[1].split("# %% Cell 4")[0], "cell3", "exec"))

In [ ]:
import torch
# ARM salah satu dari:
#   A1rT_fixed_ABS_KP_TYPELEAN | A1rT_fixed_ABS_KP_TYPELEAN_EVOBS | A1rT_fixed_ABS_KP_TYPE | A1rT_fixed_ABS_KP_EVOBS
#   A1rT_fixed_ABS_KP_NOSPEC | A1rT_fixed_ABS_KP_P1 | A1rT_fixed_ABS_KP
ARM = "A1rT_fixed_ABS_KP_TYPELEAN"
CASES = DEV30                 # DEV30 | SUBSET8 | HELDOUT | ALL_IDS
SHARD = None                  # None, atau (k, n) dengan k mulai dari 0
TAG = "dev30"
assert ARM in ARMS, f"arm tidak ada di kode yang di-clone ({COMMIT}); sudah git push? arms: {sorted(ARMS)}"
if SHARD:
    CASES = [c for i, c in enumerate(CASES) if i % SHARD[1] == SHARD[0]]
print(f"{ARM}: {len(CASES)} cases | commit {COMMIT} | GPUs free GB:", [round(torch.cuda.mem_get_info(i)[0] / 1e9, 1) for i in range(torch.cuda.device_count())])

In [ ]:
# Cell 5: dibaca dari repo hasil clone, jadi selalu sesuai commit (arm baru tidak perlu notebook baru)
src = open(os.path.join(WORK, "scripts", "kaggle_run.py"), encoding="utf-8").read()
exec(compile(src.split("# %% Cell 5")[1].split("\n", 1)[1].split("# %% Cell 6")[0], "cell5", "exec"))

In [ ]:
# Cell 6: dibaca dari repo hasil clone, jadi selalu sesuai commit (arm baru tidak perlu notebook baru)
src = open(os.path.join(WORK, "scripts", "kaggle_run.py"), encoding="utf-8").read()
exec(compile(src.split("# %% Cell 6")[1].split("\n", 1)[1], "cell6", "exec"))